<a href="https://colab.research.google.com/github/manahilatif/roadmapify/blob/feature%2Fyt_api/Youtube.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall youtube-transcript-api -y
!pip install youtube-transcript-api

# Verify the import works correctly
from youtube_transcript_api import YouTubeTranscriptApi
print("Library imported successfully!")
print("Available methods:", dir(YouTubeTranscriptApi))

# Test with a simple video
try:
    test = YouTubeTranscriptApi.get_transcript("dQw4w9WgXcQ")
    print(f"✅ Test successful! Got {len(test)} segments")
except Exception as e:
    print(f"Test failed: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 9.1 MB/s eta 0:00:00
Library imported successfully!
Available methods: ['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'fetch', 'list']
Test failed: type object 'YouTubeTranscriptApi' has no attribute 'get_transcript'


In [ ]:
# ============================================================
# STEP 1 — FETCH & SAVE ORIGINAL DATASET (FULLY WORKING)
# ============================================================
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
import os
import json

# ─────────────────────────────────────────────
# WORKING VIDEOS (ALL HAVE TRANSCRIPTS)
# These are videos that definitely have captions
# ─────────────────────────────────────────────
VIDEOS = [
    {
        "id"   : "PkZNo7MFNFg",  # Learn JavaScript - Full Course (freeCodeCamp) - HAS CAPTIONS
        "title": "Learn JavaScript - Full Course for Beginners - freeCodeCamp",
        "topic": "web_development"
    },
    {
        "id"   : "DLX62G4lc44",  # React JS Full Course (freeCodeCamp) - HAS CAPTIONS
        "title": "React JS Full Course for Beginners - freeCodeCamp",
        "topic": "web_development"
    },
    {
        "id"   : "W6NZfCO5SIk",  # CSS Full Course (freeCodeCamp) - HAS CAPTIONS
        "title": "CSS Full Course for Beginners - freeCodeCamp",
        "topic": "web_development"
    },

    # ── DATA SCIENCE (3 videos with captions) ──────────────────
    {
        "id"   : "LHBE6Q9XlzI",  # Python for Data Science (IBM) - HAS CAPTIONS
        "title": "Python for Data Science - IBM",
        "topic": "data_science"
    },

    # ── UI/UX DESIGN (3 videos with captions) ──────────────────
    {
        "id"   : "c9Wg6Cb_YlU",  # UI UX Design Tutorial - HAS CAPTIONS
        "title": "UI UX Design Tutorial for Beginners",
        "topic": "ui_ux"
    },
]

# ─────────────────────────────────────────────────────────────
# OUTPUT FOLDER
# ─────────────────────────────────────────────────────────────
ORIGINAL_DIR = "original_dataset"
os.makedirs(ORIGINAL_DIR, exist_ok=True)


# ─────────────────────────────────────────────────────────────
# HELPER: Convert a video title into a safe filename
# ─────────────────────────────────────────────────────────────
def title_to_filename(title):
    safe = title.strip()
    safe = safe.replace(" ", "_")
    safe = "".join(c for c in safe if c not in r'/\:*?"<>|')
    return safe + ".txt"


# ─────────────────────────────────────────────────────────────
# FETCH transcript from YouTube (FIXED for FetchedTranscriptSnippet)
# ─────────────────────────────────────────────────────────────
def fetch_transcript(video):
    """
    Downloads the transcript for one video.
    """
    video_id = video["id"]
    title    = video["title"]
    topic    = video["topic"]

    print(f"\n📥 Fetching  : {title}")
    print(f"   Topic     : {topic}")
    print(f"   Video ID  : {video_id}")

    try:
        # Create instance and fetch transcript
        yt_api = YouTubeTranscriptApi()
        transcript_data = yt_api.fetch(video_id)

        # Handle the FetchedTranscriptSnippet objects correctly
        # Convert to list of dictionaries with text, start, duration
        transcript_list = []
        for item in transcript_data:
            transcript_list.append({
                "text": item.text,
                "start": item.start,
                "duration": item.duration
            })

        # Merge all caption segments into one continuous paragraph
        full_text = " ".join([entry["text"] for entry in transcript_list])

        print(f"   ✅ Success : {len(transcript_list)} segments | "
              f"{len(full_text):,} characters")

        return {
            "video_id"      : video_id,
            "title"         : title,
            "topic"         : topic,
            "total_segments": len(transcript_list),
            "total_chars"   : len(full_text),
            "raw_text"      : full_text,
            "status"        : "success"
        }

    except TranscriptsDisabled:
        print(f"   ❌ Transcripts are disabled for this video")
        return {"video_id": video_id, "title": title, "topic": topic,
                "total_segments": 0, "total_chars": 0,
                "status": "transcripts_disabled", "raw_text": ""}

    except NoTranscriptFound:
        print(f"   ❌ No transcript available for this video")
        return {"video_id": video_id, "title": title, "topic": topic,
                "total_segments": 0, "total_chars": 0,
                "status": "no_transcript_found", "raw_text": ""}

    except Exception as e:
        print(f"   ❌ Unexpected error: {e}")
        return {"video_id": video_id, "title": title, "topic": topic,
                "total_segments": 0, "total_chars": 0,
                "status": f"error: {str(e)}", "raw_text": ""}


# ─────────────────────────────────────────────────────────────
# SAVE original transcript file
# ─────────────────────────────────────────────────────────────
def save_original(result):
    """
    Saves the raw transcript to original_dataset/<Video_Title>.txt
    """
    if not result["raw_text"]:
        print(f"   ⚠️  No text to save — skipping")
        return None

    filename = title_to_filename(result["title"])
    filepath = os.path.join(ORIGINAL_DIR, filename)

    if os.path.exists(filepath):
        print(f"   ⚠️  Original already exists — NOT overwriting: {filepath}")
        return filepath

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"TITLE    : {result['title']}\n")
        f.write(f"TOPIC    : {result['topic']}\n")
        f.write(f"VIDEO ID : {result['video_id']}\n")
        f.write(f"SEGMENTS : {result['total_segments']}\n")
        f.write(f"CHARS    : {result['total_chars']}\n")
        f.write("=" * 60 + "\n\n")
        f.write(result["raw_text"])

    print(f"   💾 Saved original → {filepath}")
    return filepath


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("  STEP 1 — FETCHING & SAVING ORIGINAL DATASET")
    print("=" * 60)
    print(f"  Output folder : {ORIGINAL_DIR}/")
    print(f"  Total videos  : {len(VIDEOS)}")
    print("=" * 60)

    all_results = []
    saved_files = []

    for video in VIDEOS:
        result   = fetch_transcript(video)
        filepath = save_original(result)
        all_results.append(result)
        if filepath:
            saved_files.append(filepath)

    # Save fetch summary
    summary_path = os.path.join(ORIGINAL_DIR, "_fetch_summary.json")
    summary_data = [
        {k: v for k, v in r.items() if k != "raw_text"}
        for r in all_results
    ]
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary_data, f, indent=2)

    # Final report
    success = sum(1 for r in all_results if r["status"] == "success")
    failed  = len(all_results) - success

    print("\n" + "=" * 60)
    print("  STEP 1 COMPLETE")
    print(f"  ✅ {success} originals saved")
    print(f"  ❌ {failed} failed")
    print(f"\n  Original files in: {ORIGINAL_DIR}/")
    for fp in saved_files:
        print(f"    → {os.path.basename(fp)}")
    print("\n  ⚠️  NEVER edit files inside original_dataset/")
    print("  Next → run step2_clean.py")
    print("=" * 60)


if __name__ == "__main__":
    main()

  STEP 1 — FETCHING & SAVING ORIGINAL DATASET
  Output folder : original_dataset/
  Total videos  : 5

📥 Fetching  : Learn JavaScript - Full Course for Beginners - freeCodeCamp
   Topic     : web_development
   Video ID  : PkZNo7MFNFg
   ✅ Success : 2996 segments | 156,407 characters
   💾 Saved original → original_dataset/Learn_JavaScript_-_Full_Course_for_Beginners_-_freeCodeCamp.txt

📥 Fetching  : React JS Full Course for Beginners - freeCodeCamp
   Topic     : web_development
   Video ID  : DLX62G4lc44
   ✅ Success : 3174 segments | 267,654 characters
   💾 Saved original → original_dataset/React_JS_Full_Course_for_Beginners_-_freeCodeCamp.txt

📥 Fetching  : CSS Full Course for Beginners - freeCodeCamp
   Topic     : web_development
   Video ID  : W6NZfCO5SIk
   ✅ Success : 524 segments | 38,881 characters
   💾 Saved original → original_dataset/CSS_Full_Course_for_Beginners_-_freeCodeCamp.txt

📥 Fetching  : Python for Data Science - IBM
   Topic     : data_science
   Video ID  : LHBE

In [ ]:
# ============================================================
# STEP 2 — CLEAN THE TRANSCRIPTS (SEPARATE DATASET)
# ============================================================
# GOLDEN RULE:
#   This script only READS from original_dataset/.
#   It NEVER writes back to original_dataset/.
#   All cleaned output goes to clean_dataset/ — a completely
#   separate folder with separate files.
#
# Input  folder : original_dataset/   ← READ ONLY, never touched
# Output folder : clean_dataset/      ← All cleaned files go here
#
# For each original file, two clean outputs are created:
#   1. clean_dataset/<Video_Title>_CLEAN.txt   (individual file)
#   2. clean_dataset/all_transcripts_clean.csv (combined dataset)
# ============================================================

import os
import re
import csv

# ─────────────────────────────────────────────────────────────
# FOLDERS
# ─────────────────────────────────────────────────────────────
ORIGINAL_DIR = "original_dataset"   # READ ONLY — never write here
CLEAN_DIR    = "clean_dataset"      # all cleaned output goes here
os.makedirs(CLEAN_DIR, exist_ok=True)


# ─────────────────────────────────────────────────────────────
# THE CLEANING PIPELINE — 7 steps
# ─────────────────────────────────────────────────────────────
def clean_text(raw_text):
    """
    Cleans a raw YouTube transcript through 7 steps.
    Input  : raw string exactly as saved by step1
    Output : cleaned string (does NOT modify the input variable)
    """

    # Work on a copy so the original string is never mutated
    text = raw_text

    # ── Step 1: Remove YouTube auto-caption noise ─────────────
    # YouTube's auto-captions add tags like [Music], [Applause],
    # [Laughter], (music), (crowd cheering) — none of these are
    # spoken words, so they are removed.
    text = re.sub(r'\[.*?\]', '', text)    # removes [Music], [Applause]
    text = re.sub(r'\(.*?\)', '', text)    # removes (music), (laughter)

    # ── Step 2: Remove URLs ───────────────────────────────────
    # Speakers sometimes say "visit http://..." — remove the URL part.
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # ── Step 3: Remove special characters ────────────────────
    # Keep only: letters (a-z A-Z), digits (0-9), spaces,
    #            commas, periods, apostrophes, ! and ?
    # This removes: @, #, $, %, ^, &, *, emojis, dashes, etc.
    text = re.sub(r"[^a-zA-Z0-9\s,.'!?]", '', text)

    # ── Step 4: Remove filler words ──────────────────────────
    # Auto-captions often include speech fillers: uh, um, hmm, etc.
    # \b = word boundary so "hum" in "human" is NOT removed.
    text = re.sub(r'\b(uh|um|hmm|erm|ah|oh)\b', '', text, flags=re.IGNORECASE)

    # ── Step 5: Normalize whitespace ─────────────────────────
    # After removals above, there may be multiple consecutive spaces.
    # Replace any run of whitespace (spaces, tabs, newlines) with one space.
    text = re.sub(r'\s+', ' ', text)

    # ── Step 6: Strip leading and trailing whitespace ─────────
    text = text.strip()

    # ── Step 7: Convert to lowercase ─────────────────────────
    # Uniform casing for consistent text analysis later.
    text = text.lower()

    return text


# ─────────────────────────────────────────────────────────────
# PARSE an original file into metadata + raw transcript text
# ─────────────────────────────────────────────────────────────
def parse_original_file(filepath):
    """
    Reads one file from original_dataset/.
    Splits it into:
      - metadata dict  (TITLE, TOPIC, VIDEO ID, SEGMENTS, CHARS)
      - raw_text str   (the actual transcript, below the === line)

    Does NOT modify the file in any way.
    """
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    lines            = content.split("\n")
    metadata         = {}
    text_start_index = 0

    # Header is always the first 5 lines, then a === separator
    for i, line in enumerate(lines[:7]):
        if   line.startswith("TITLE"):
            metadata["title"]    = line.split(":", 1)[1].strip()
        elif line.startswith("TOPIC"):
            metadata["topic"]    = line.split(":", 1)[1].strip()
        elif line.startswith("VIDEO ID"):
            metadata["video_id"] = line.split(":", 1)[1].strip()
        elif line.startswith("SEGMENTS"):
            metadata["segments"] = line.split(":", 1)[1].strip()
        elif line.startswith("CHARS"):
            metadata["chars"]    = line.split(":", 1)[1].strip()
        elif line.startswith("=" * 10):
            # Transcript text begins two lines after the === separator
            text_start_index = i + 2
            break

    raw_text = "\n".join(lines[text_start_index:]).strip()
    return metadata, raw_text


# ─────────────────────────────────────────────────────────────
# SAVE a single clean file named after the video title
# ─────────────────────────────────────────────────────────────
def save_clean_file(metadata, cleaned_text, original_filename, stats):
    """
    Saves the cleaned transcript to:
        clean_dataset/<Video_Title>_CLEAN.txt

    The filename mirrors the original but adds _CLEAN before .txt
    so the two are easy to compare side by side.

    File layout:
        TITLE         : <video title>
        TOPIC         : <topic>
        VIDEO ID      : <video id>
        ORIGINAL FILE : <name of the source original file>
        RAW WORDS     : <word count before cleaning>
        CLEAN WORDS   : <word count after cleaning>
        WORDS REMOVED : <difference>
        ============================================================
        <cleaned transcript text>
    """
    # Build output filename: add _CLEAN before .txt
    base        = original_filename.replace(".txt", "")
    out_name    = base + "_CLEAN.txt"
    out_path    = os.path.join(CLEAN_DIR, out_name)

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(f"TITLE         : {metadata.get('title', '')}\n")
        f.write(f"TOPIC         : {metadata.get('topic', '')}\n")
        f.write(f"VIDEO ID      : {metadata.get('video_id', '')}\n")
        f.write(f"ORIGINAL FILE : {original_filename}\n")
        f.write(f"RAW WORDS     : {stats['raw_words']:,}\n")
        f.write(f"CLEAN WORDS   : {stats['clean_words']:,}\n")
        f.write(f"WORDS REMOVED : {stats['removed']:,} "
                f"({stats['removed_pct']:.1f}%)\n")
        f.write("=" * 60 + "\n\n")
        f.write(cleaned_text)

    return out_path


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("  STEP 2 — CLEANING TRANSCRIPTS")
    print("=" * 60)
    print(f"  Reading from  : {ORIGINAL_DIR}/   ← READ ONLY")
    print(f"  Writing to    : {CLEAN_DIR}/")
    print("=" * 60)

    # Collect all .txt files from original_dataset/
    # Skip the _fetch_summary.json and any hidden files
    original_files = sorted([
        f for f in os.listdir(ORIGINAL_DIR)
        if f.endswith(".txt") and not f.startswith("_")
    ])

    if not original_files:
        print("❌ No files found in original_dataset/")
        print("   Run step1_fetch_transcripts.py first!")
        return

    print(f"📂 Found {len(original_files)} original file(s)\n")

    csv_rows    = []   # collects data for the combined CSV
    clean_files = []   # tracks paths of saved clean files

    for original_filename in original_files:
        original_path = os.path.join(ORIGINAL_DIR, original_filename)

        print(f"🧹 Processing : {original_filename}")

        # ── Parse the original (READ ONLY) ───────────────────
        metadata, raw_text = parse_original_file(original_path)

        if not raw_text:
            print(f"   ⚠️  Empty transcript — skipping\n")
            continue

        # ── Apply cleaning pipeline ───────────────────────────
        cleaned_text = clean_text(raw_text)

        # ── Compute statistics ────────────────────────────────
        raw_words   = len(raw_text.split())
        clean_words = len(cleaned_text.split())
        removed     = raw_words - clean_words
        removed_pct = (removed / max(raw_words, 1)) * 100

        stats = {
            "raw_words"  : raw_words,
            "clean_words": clean_words,
            "removed"    : removed,
            "removed_pct": removed_pct
        }

        print(f"   Raw text   : {raw_words:,} words")
        print(f"   Clean text : {clean_words:,} words")
        print(f"   Removed    : {removed:,} words ({removed_pct:.1f}%)")

        # ── Save individual clean file ────────────────────────
        out_path = save_clean_file(metadata, cleaned_text,
                                   original_filename, stats)
        clean_files.append(out_path)
        print(f"   💾 Clean file → {out_path}\n")

        # ── Collect row for the combined CSV ──────────────────
        csv_rows.append({
            "original_file" : original_filename,
            "video_id"      : metadata.get("video_id", ""),
            "title"         : metadata.get("title", ""),
            "topic"         : metadata.get("topic", ""),
            "raw_words"     : raw_words,
            "clean_words"   : clean_words,
            "words_removed" : removed,
            "removed_pct"   : f"{removed_pct:.1f}%",
            "clean_text"    : cleaned_text
        })

    # ── Save combined CSV ─────────────────────────────────────
    # This is the "master clean dataset" — one row per video
    csv_path = os.path.join(CLEAN_DIR, "all_transcripts_clean.csv")
    fieldnames = [
        "original_file", "video_id", "title", "topic",
        "raw_words", "clean_words", "words_removed",
        "removed_pct", "clean_text"
    ]
    with open(csv_path, "w", encoding="utf-8", newline="") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(csv_rows)

    # ── Final report ──────────────────────────────────────────
    print("=" * 60)
    print("  STEP 2 COMPLETE")
    print(f"  ✅ {len(clean_files)} individual clean files saved")
    print(f"  📄 Combined CSV → {csv_path}")
    print(f"\n  original_dataset/ — unchanged ✓ (never modified)")
    print(f"  clean_dataset/    — all cleaned output here")
    print("\n  Next → run step3_verify.py to confirm everything is correct")
    print("=" * 60)


if __name__ == "__main__":
    main()


  STEP 2 — CLEANING TRANSCRIPTS
  Reading from  : original_dataset/   ← READ ONLY
  Writing to    : clean_dataset/
📂 Found 5 original file(s)

🧹 Processing : CSS_Full_Course_for_Beginners_-_freeCodeCamp.txt
   Raw text   : 7,352 words
   Clean text : 7,349 words
   Removed    : 3 words (0.0%)
   💾 Clean file → clean_dataset/CSS_Full_Course_for_Beginners_-_freeCodeCamp_CLEAN.txt

🧹 Processing : Learn_JavaScript_-_Full_Course_for_Beginners_-_freeCodeCamp.txt
   Raw text   : 29,450 words
   Clean text : 28,530 words
   Removed    : 920 words (3.1%)
   💾 Clean file → clean_dataset/Learn_JavaScript_-_Full_Course_for_Beginners_-_freeCodeCamp_CLEAN.txt

🧹 Processing : Python_for_Data_Science_-_IBM.txt
   Raw text   : 106,748 words
   Clean text : 106,732 words
   Removed    : 16 words (0.0%)
   💾 Clean file → clean_dataset/Python_for_Data_Science_-_IBM_CLEAN.txt

🧹 Processing : React_JS_Full_Course_for_Beginners_-_freeCodeCamp.txt
   Raw text   : 50,579 words
   Clean text : 50,579 words
   R